# DSPy Learning Examples - Comprehensive Jupyter Notes

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duksaramio/dspy-jupyter-notes/blob/main/dspy_learn_examples.ipynb)

This notebook contains comprehensive examples from the [DSPy Learn](https://dspy.ai/learn/) documentation, organized by topic:

## Table of Contents
1. [Language Models](#language-models) - Setting up and configuring LMs
2. [Signatures](#signatures) - Defining input/output behavior
3. [Modules](#modules) - Using DSPy modules like Predict, ChainOfThought
4. [Adapters](#adapters) - ChatAdapter and JSONAdapter
5. [Tools](#tools) - Using tools with ReAct and manual handling
6. [MCP](#mcp) - Model Context Protocol integration
7. [Data Handling](#data-handling) - Working with Example objects
8. [Metrics](#metrics) - Defining evaluation metrics
9. [Optimizers](#optimizers) - Optimizing prompts and weights

## How to Run in Google Colab

1. Click the Colab button above, OR
2. Go to [Google Colab](https://colab.research.google.com)
3. Click "GitHub" tab
4. Enter: `https://github.com/duksaramio/dspy-jupyter-notes`
5. Select `dspy_learn_examples.ipynb`
6. Update your API key in the Setup section
7. Run the cells!


## Setup

First, let's install DSPy and set up the environment.

In [ ]:
# Install DSPy
!pip install -U dspy

In [ ]:
import dspy
import os

# Set up your API key (replace with your own)
# Option 1: OpenAI
os.environ['OPENAI_API_KEY'] = 'your-openai-api-key-here'

# Option 2: Anthropic (Claude)
os.environ['ANTHROPIC_API_KEY'] = 'your-anthropic-api-key-here'

# Configure DSPy with a language model
lm = dspy.LM('openai/gpt-4o-mini')
dspy.configure(lm=lm)

---

<a id="language-models"></a>
# 1. Language Models

Learn how to configure and use Language Models in DSPy.

### 1.1 Basic LM Setup

In [ ]:
# Basic LM setup with API key
import dspy

lm = dspy.LM('openai/gpt-4o-mini', api_key='YOUR_OPENAI_API_KEY')
dspy.configure(lm=lm)

### 1.2 Calling the LM Directly

In [ ]:
# Call LM directly with a simple prompt
lm("Say this is a test!", temperature=0.7)

In [ ]:
# Call LM directly with messages
lm(messages=[{"role": "user", "content": "Say this is a test!"}])

### 1.3 Using LM with DSPy Modules

In [ ]:
# Define a module (ChainOfThought) and assign it a signature
qa = dspy.ChainOfThought('question -> answer')

# Run with the default LM
response = qa(question="How many floors are in the castle David Gregory inherited?")
print(response.answer)

### 1.4 Using Multiple LMs

In [ ]:
# Configure global LM
dspy.configure(lm=dspy.LM('openai/gpt-4o-mini'))
response = qa(question="How many floors are in the castle David Gregory inherited?")
print('GPT-4o-mini:', response.answer)

# Use context to switch LM locally
with dspy.context(lm=dspy.LM('openai/gpt-3.5-turbo')):
    response = qa(question="How many floors are in the castle David Gregory inherited?")
    print('GPT-3.5-turbo:', response.answer)

### 1.5 Configuring LM Generation

In [ ]:
# Configure LM with generation parameters
gpt_4o_mini = dspy.LM('openai/gpt-4o-mini', temperature=0.9, max_tokens=3000, stop=None, cache=False)

In [ ]:
# Using rollout_id for diverse outputs
lm("Say this is a test!", rollout_id=1, temperature=1.0)

In [ ]:
# Configure at module level
predict = dspy.Predict("question -> answer", rollout_id=1, temperature=1.0)

In [ ]:
# Override for single invocation
predict = dspy.Predict("question -> answer")
predict(question="What is 1 + 52?", config={"rollout_id": 5, "temperature": 1.0})

### 1.6 Inspecting Output and Usage Metadata

In [ ]:
# Check LM history
len(lm.history)  # Number of calls to the LM
lm.history[-1].keys()  # Access the last call with all metadata

### 1.7 Using the Responses API

## Anthropic (Claude) API

Learn how to use Anthropic's Claude models with DSPy.

### Anthropic (Claude) Setup

In [ ]:
# Set your Anthropic API key
import os
os.environ['ANTHROPIC_API_KEY'] = 'your-anthropic-api-key'

# Configure DSPy with Claude
lm = dspy.LM('anthropic/claude-3-5-sonnet-20241022')
dspy.configure(lm=lm)

### Using Claude with DSPy Modules

In [ ]:
# Define a module with Claude
qa = dspy.ChainOfThought('question -> answer')

# Run with Claude
response = qa(question="What is the capital of France?")
print(response.answer)

In [ ]:
# Configure DSPy to use the Responses API
dspy.configure(
    lm=dspy.LM(
        "openai/gpt-5-mini",
        model_type="responses",
        temperature=1.0,
        max_tokens=16000,
    ),
)

---

<a id="signatures"></a>
# 2. Signatures

Signatures define the input/output behavior of DSPy modules.

### 2.1 Inline DSPy Signatures

In [ ]:
# Basic inline signatures
# Question Answering: "question -> answer"
# Sentiment Classification: "sentence -> sentiment: bool"
# Summarization: "document -> summary"
# Retrieval-Augmented QA: "context: list[str], question: str -> answer: str"
# Multiple-Choice with Reasoning: "question, choices: list[str] -> reasoning: str, selection: int"

### 2.2 Inline Signature with Instructions

In [ ]:
# Inline signature with custom instructions
toxicity = dspy.Predict(
    dspy.Signature(
        "comment -> toxic: bool",
        instructions="Mark as 'toxic' if the comment includes insults, harassment, or sarcastic derogatory remarks.",
    )
)
comment = "you are beautiful."
toxicity(comment=comment).toxic

### 2.3 Example A: Sentiment Classification

In [ ]:
# Sentiment Classification example
sentence = "it's a charming and often affecting journey."
classify = dspy.Predict('sentence -> sentiment: bool')
classify(sentence=sentence).sentiment

### 2.4 Example B: Summarization with ChainOfThought

In [ ]:
# Summarization example with ChainOfThought
document = """The 21-year-old made seven appearances for the Hammers and netted his only goal for them in a Europa League qualification round match against Andorran side FC Lustrains last season. Lee had two loan spells in League One last term, with Blackpool and then Colchester United. He scored twice for the U's but was unable to save them from relegation. The length of Lee's contract with the promoted Tykes has not been revealed. Find all the latest football transfers on our dedicated page."""

summarize = dspy.ChainOfThought('document -> summary')
response = summarize(document=document)
print(response.summary)
print("Reasoning:", response.reasoning)

### 2.5 Class-based DSPy Signatures

In [ ]:
# Example C: Classification with Literal types
from typing import Literal

class Emotion(dspy.Signature):
    """Classify emotion."""
    sentence: str = dspy.InputField()
    sentiment: Literal['sadness', 'joy', 'love', 'anger', 'fear', 'surprise'] = dspy.OutputField()

sentence = "i started feeling a little vulnerable when the giant spotlight started blinding me"
classify = dspy.Predict(Emotion)
classify(sentence=sentence)

### 2.6 Example D: Citation Faithfulness Metric

In [ ]:
# A metric that evaluates faithfulness to citations
class CheckCitationFaithfulness(dspy.Signature):
    """Verify that the text is based on the provided context."""
    context: str = dspy.InputField(desc="facts here are assumed to be true")
    text: str = dspy.InputField()
    faithfulness: bool = dspy.OutputField()
    evidence: dict[str, list[str]] = dspy.OutputField(desc="Supporting evidence for claims")

context = "The 21-year-old made seven appearances for the Hammers and netted his only goal for them in a Europa League qualification round match against Andorran side FC Lustrains last season. Lee had two loan spells in League One last term, with Blackpool and then Colchester United. He scored twice for the U's but was unable to save them from relegation. The length of Lee's contract with the promoted Tykes has not been revealed."
text = "Lee scored 3 goals for Colchester United."
faithfulness = dspy.ChainOfThought(CheckCitationFaithfulness)
faithfulness(context=context, text=text)

### 2.7 Example E: Multi-modal Image Classification

In [ ]:
# Multi-modal image classification
class DogPictureSignature(dspy.Signature):
    """Output the dog breed of the dog in the image."""
    image_1: dspy.Image = dspy.InputField(desc="An image of a dog")
    answer: str = dspy.OutputField(desc="The dog breed of the dog in the image")

image_url = "https://picsum.photos/id/237/200/300"
classify = dspy.Predict(DogPictureSignature)
classify(image_1=dspy.Image.from_url(image_url))

### 2.8 Working with Custom Types

In [ ]:
# Simple custom type
import pydantic

class QueryResult(pydantic.BaseModel):
    text: str
    score: float

signature = dspy.Signature("query: str -> result: QueryResult")

# Nested custom types
class MyContainer:
    class Query(pydantic.BaseModel):
        text: str
    class Score(pydantic.BaseModel):
        score: float

signature = dspy.Signature("query: MyContainer.Query -> score: MyContainer.Score")

---

<a id="modules"></a>
# 3. Modules

DSPy modules are building blocks for programs that use LMs.

### 3.1 Using dspy.Predict

In [ ]:
# Using dspy.Predict module
sentence = "it's a charming and often affecting journey."

# 1) Declare with a signature
classify = dspy.Predict('sentence -> sentiment: bool')

# 2) Call with input argument(s)
response = classify(sentence=sentence)

# 3) Access the output
print(response.sentiment)

### 3.2 Using dspy.ChainOfThought

In [ ]:
# Using dspy.ChainOfThought
question = "What's something great about the ColBERT retrieval model?"

# 1) Declare with a signature, and pass some config
classify = dspy.ChainOfThought('question -> answer', temperature=0.7)

# 2) Call with input argument
response = classify(question=question)

# 3) Access the output
response.answer

In [ ]:
# Inspect reasoning and answer
print(f"Reasoning: {response.reasoning}")
print(f"Answer: {response.answer}")

### 3.3 Other DSPy Modules

In [ ]:
# Math example with ChainOfThought
math = dspy.ChainOfThought("question -> answer: float")
math(question="Two dice are tossed. What is the probability that the sum equals two?")

### 3.4 Composing Multiple Modules

In [ ]:
# Custom module composition example (multi-hop search)
class Hop(dspy.Module):
    def __init__(self, num_docs=10, num_hops=4):
        self.num_docs, self.num_hops = num_docs, num_hops
        self.generate_query = dspy.ChainOfThought('claim, notes -> query')
        self.append_notes = dspy.ChainOfThought('claim, notes, context -> new_notes: list[str], titles: list[str]')

    def forward(self, claim: str) -> list[str]:
        notes = []
        titles = []
        for _ in range(self.num_hops):
            query = self.generate_query(claim=claim, notes=notes).query
            # context = search(query, k=self.num_docs)  # Assume search is defined
            # prediction = self.append_notes(claim=claim, notes=notes, context=context)
            # notes.extend(prediction.new_notes)
            # titles.extend(prediction.titles)
        return dspy.Prediction(notes=notes, titles=list(set(titles)))

# Create and use the custom module
hop = Hop()
# hop(claim="Stephen Curry is the best 3 pointer shooter ever in the human history")

### 3.5 Tracking LM Usage

In [ ]:
# Enable usage tracking
dspy.configure(track_usage=True)

# Define a simple program
class MyProgram(dspy.Module):
    def __init__(self):
        self.predict1 = dspy.ChainOfThought("question -> answer")
        self.predict2 = dspy.ChainOfThought("question, answer -> score")

    def __call__(self, question: str) -> str:
        answer = self.predict1(question=question)
        score = self.predict2(question=question, answer=answer)
        return score

# Run the program and check usage
program = MyProgram()
output = program(question="What is the capital of France?")
print(output.get_lm_usage())

---

<a id="adapters"></a>
# 4. Adapters

Adapters bridge DSPy modules and Language Models.

### 4.1 Configure Adapters

In [ ]:
# Default configuration (uses ChatAdapter)
dspy.configure(lm=dspy.LM("openai/gpt-4o-mini"))
predict = dspy.Predict("question -> answer")
result = predict(question="What is the capital of France?")

In [ ]:
# Explicit ChatAdapter configuration
dspy.configure(
    lm=dspy.LM("openai/gpt-4o-mini"),
    adapter=dspy.ChatAdapter(),  # This is the default
)
predict = dspy.Predict("question -> answer")
result = predict(question="What is the capital of France?")

### 4.2 Using ChatAdapter

In [ ]:
# View formatted messages
signature = dspy.Signature("question -> answer")
inputs = {"question": "What is 2+2?"}
demos = [{"question": "What is 1+1?", "answer": "2"}]
adapter = dspy.ChatAdapter()
print(adapter.format(signature, demos, inputs))

In [ ]:
# Fetch system message only
signature = dspy.Signature("question -> answer")
system_message = dspy.ChatAdapter().format_system_message(signature)
print(system_message)

### 4.3 Using ChatAdapter with Pydantic Models

In [ ]:
# ChatAdapter with structured output
import pydantic

dspy.configure(lm=dspy.LM("openai/gpt-4o-mini"), adapter=dspy.ChatAdapter())

class ScienceNews(pydantic.BaseModel):
    text: str
    scientists_involved: list[str]

class NewsQA(dspy.Signature):
    """Get news about the given science field"""
    science_field: str = dspy.InputField()
    year: int = dspy.InputField()
    num_of_outputs: int = dspy.InputField()
    news: list[ScienceNews] = dspy.OutputField(desc="science news")

predict = dspy.Predict(NewsQA)
predict(science_field="Computer Theory", year=2022, num_of_outputs=1)
# dspy.inspect_history()  # View full history

### 4.4 Using JSONAdapter

In [ ]:
# JSONAdapter configuration
dspy.configure(lm=dspy.LM("openai/gpt-4o-mini"), adapter=dspy.JSONAdapter())

class ScienceNews(pydantic.BaseModel):
    text: str
    scientists_involved: list[str]

class NewsQA(dspy.Signature):
    """Get news about the given science field"""
    science_field: str = dspy.InputField()
    year: int = dspy.InputField()
    num_of_outputs: int = dspy.InputField()
    news: list[ScienceNews] = dspy.OutputField(desc="science news")

predict = dspy.Predict(NewsQA)
predict(science_field="Computer Theory", year=2022, num_of_outputs=1)
# dspy.inspect_history()  # View full history

### 4.5 Adapter Configuration Options

In [ ]:
# ChatAdapter with native function calling enabled
chat_adapter_native = dspy.ChatAdapter(use_native_function_calling=True)

# JSONAdapter with native function calling disabled
json_adapter_manual = dspy.JSONAdapter(use_native_function_calling=False)

# Configure DSPy
dspy.configure(lm=dspy.LM(model="openai/gpt-4o"), adapter=chat_adapter_native)

---

<a id="tools"></a>
# 5. Tools

DSPy provides support for tool-using agents.

### 5.1 Using dspy.ReAct (Fully Managed)

In [ ]:
# ReAct agent example
import dspy

# Define your tools as functions
def get_weather(city: str) -> str:
    """Get the current weather for a city."""
    return f"The weather in {city} is sunny and 75°F"

def search_web(query: str) -> str:
    """Search the web for information."""
    return f"Search results for '{query}': [relevant information...]"

# Create a ReAct agent
react_agent = dspy.ReAct(
    signature="question -> answer",
    tools=[get_weather, search_web],
    max_iters=5
)

# Use the agent
result = react_agent(question="What's the weather like in Tokyo?")
print(result.answer)
print("Tool calls made:", result.trajectory)

### 5.2 ReAct Parameters

In [ ]:
# ReAct with parameters
react_agent = dspy.ReAct(
    signature="question -> answer",  # Input/output specification
    tools=[tool1, tool2, tool3],  # List of available tools
    max_iters=10  # Maximum number of tool call iterations
)

### 5.3 Manual Tool Handling - Basic Setup

In [ ]:
# Manual tool handling
import dspy

class ToolSignature(dspy.Signature):
    """Signature for manual tool handling."""
    question: str = dspy.InputField()
    tools: list[dspy.Tool] = dspy.InputField()
    outputs: dspy.ToolCalls = dspy.OutputField()

def weather(city: str) -> str:
    """Get weather information for a city."""
    return f"The weather in {city} is sunny"

def calculator(expression: str) -> str:
    """Evaluate a mathematical expression."""
    try:
        result = eval(expression)
        return f"The result is {result}"
    except:
        return "Invalid expression"

# Create tool instances
tools = {
    "weather": dspy.Tool(weather),
    "calculator": dspy.Tool(calculator)
}

# Create predictor
predictor = dspy.Predict(ToolSignature)

# Make a prediction
response = predictor(
    question="What's the weather in New York?",
    tools=list(tools.values())
)

# Execute the tool calls
for call in response.outputs.tool_calls:
    result = call.execute()
    print(f"Tool: {call.name}")
    print(f"Args: {call.args}")
    print(f"Result: {result}")

### 5.4 Understanding dspy.Tool

In [ ]:
# Using dspy.Tool
def my_function(param1: str, param2: int = 5) -> str:
    """A sample function with parameters."""
    return f"Processed {param1} with value {param2}"

# Create a tool
tool = dspy.Tool(my_function)

# Tool properties
print(tool.name)  # "my_function"
print(tool.desc)  # The function's docstring
print(tool.args)  # Parameter schema
print(str(tool))  # Full tool description

### 5.5 Understanding dspy.ToolCalls

In [ ]:
# Working with ToolCalls
for call in response.outputs.tool_calls:
    print(f"Tool name: {call.name}")
    print(f"Arguments: {call.args}")
    
    # Execute individual tool calls
    result = call.execute()  # Automatic discovery
    # Or with explicit functions
    result = call.execute(functions={"weather": weather, "calculator": calculator})
    print(f"Result: {result}")

### 5.6 Async Tools

In [ ]:
# Async tools with acall
import asyncio

async def async_weather(city: str) -> str:
    """Get weather information asynchronously."""
    await asyncio.sleep(0.1)
    return f"The weather in {city} is sunny"

tool = dspy.Tool(async_weather)

# Use acall for async tools
result = await tool.acall(city="New York")
print(result)

In [ ]:
# Running async tools in sync mode
with dspy.context(allow_tool_async_sync_conversion=True):
    result = tool(city="New York")
    print(result)

### 5.7 Best Practices - Tool Function Design

In [ ]:
# Good tool function design
def good_tool(city: str, units: str = "celsius") -> str:
    """
    Get weather information for a specific city.
    
    Args:
        city: The name of the city to get weather for
        units: Temperature units, either 'celsius' or 'fahrenheit'
    
    Returns:
        A string describing the current weather conditions
    """
    if not city.strip():
        return "Error: City name cannot be empty"
    return f"Weather in {city}: 25°{units[0].upper()}, sunny"

---

<a id="mcp"></a>
# 6. MCP (Model Context Protocol)

DSPy supports MCP for standardized tool integration.

### 6.1 Installation

In [ ]:
# Install DSPy with MCP support
# !pip install -U "dspy[mcp]"

### 6.2 HTTP Server (Remote)

In [ ]:
# MCP with HTTP Server (Remote)
"""
import asyncio
import dspy
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

async def main():
    # Connect to HTTP MCP server
    async with streamablehttp_client("http://localhost:8000/mcp") as (read, write):
        async with ClientSession(read, write) as session:
            # Initialize the session
            await session.initialize()
            
            # List and convert tools
            response = await session.list_tools()
            dspy_tools = [
                dspy.Tool.from_mcp_tool(session, tool)
                for tool in response.tools
            ]
            
            # Create and use ReAct agent
            class TaskSignature(dspy.Signature):
                task: str = dspy.InputField()
                result: str = dspy.OutputField()
            
            react_agent = dspy.ReAct(
                signature=TaskSignature,
                tools=dspy_tools,
                max_iters=5
            )
            
            result = await react_agent.acall(task="Check the weather in Tokyo")
            print(result.result)

asyncio.run(main())
"""

### 6.3 Stdio Server (Local Process)

In [ ]:
# MCP with Stdio Server (Local)
"""
import asyncio
import dspy
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

async def main():
    # Configure the stdio server
    server_params = StdioServerParameters(
        command="python",
        args=["path/to/your/mcp_server.py"],
        env=None,
    )
    
    # Connect to the server
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            
            # List available tools
            response = await session.list_tools()
            
            # Convert MCP tools to DSPy tools
            dspy_tools = [
                dspy.Tool.from_mcp_tool(session, tool)
                for tool in response.tools
            ]
            
            # Create a ReAct agent
            class QuestionAnswer(dspy.Signature):
                """Answer questions using available tools."""
                question: str = dspy.InputField()
                answer: str = dspy.OutputField()
            
            react_agent = dspy.ReAct(
                signature=QuestionAnswer,
                tools=dspy_tools,
                max_iters=5
            )
            
            result = await react_agent.acall(question="What is 25 + 17?")
            print(result.answer)

asyncio.run(main())
"""

---

<a id="data-handling"></a>
# 7. Data Handling

Learn how to work with Example objects in DSPy.

### 7.1 Creating Example Objects

In [ ]:
# Create Example objects
qa_pair = dspy.Example(question="This is a question?", answer="This is an answer.")

print(qa_pair)
print(qa_pair.question)
print(qa_pair.answer)

In [ ]:
# Examples with any field keys
object = dspy.Example(field1=value1, field2=value2, field3=value3)

# Training set example
trainset = [
    dspy.Example(report="LONG REPORT 1", summary="short summary 1"),
    # ... more examples
]

### 7.2 Specifying Input Keys

In [ ]:
# Mark fields as inputs
qa_pair = dspy.Example(question="This is a question?", answer="This is an answer.")

# Single Input
print(qa_pair.with_inputs("question"))

# Multiple Inputs
print(qa_pair.with_inputs("question", "answer"))

In [ ]:
# Accessing and excluding keys
article_summary = dspy.Example(
    article="This is an article.",
    summary="This is a summary."
).with_inputs("article")

input_key_only = article_summary.inputs()
non_input_key_only = article_summary.labels()

print("Example object with Input fields only:", input_key_only)
print("Example object with Non-Input fields only:", non_input_key_only)

---

<a id="metrics"></a>
# 8. Metrics

Define metrics to evaluate your DSPy programs.

### 8.1 Simple Metrics

In [ ]:
# Simple exact match metric
def validate_answer(example, pred, trace=None):
    return example.answer.lower() == pred.answer.lower()

In [ ]:
# Using built-in metrics
from dspy.evaluate import metrics

# answer_exact_match
# answer_passage_match

### 8.2 Complex Metrics

In [ ]:
# Metric checking multiple properties
def validate_context_and_answer(example, pred, trace=None):
    # Check the gold label and the predicted answer
    answer_match = example.answer.lower() == pred.answer.lower()
    
    # Check the predicted answer comes from retrieved contexts
    context_match = any((pred.answer.lower() in c) for c in pred.context) if trace is None else True
    
    if trace is None:
        # Evaluation or optimization
        return (answer_match + context_match) / 2.0
    else:
        # Bootstrapping
        return answer_match and context_match

### 8.3 Using AI Feedback for Metrics

In [ ]:
# AI-powered metric with Assess signature
class Assess(dspy.Signature):
    """Assess the quality of a tweet along the specified dimension."""
    assessed_text = dspy.InputField()
    assessment_question = dspy.InputField()
    assessment_answer: bool = dspy.OutputField()

def metric(gold, pred, trace=None):
    question, answer, tweet = gold.question, gold.answer, pred.output
    
    engaging = "Does the assessed text make for a self-contained, engaging tweet?"
    correct = f"The text should answer `{question}` with `{answer}`. Does the assessed text contain this answer?"
    
    correct = dspy.Predict(Assess)(assessed_text=tweet, assessment_question=correct)
    engaging = dspy.Predict(Assess)(assessed_text=tweet, assessment_question=engaging)
    
    correct, engaging = [m.assessment_answer for m in [correct, engaging]]
    
    if correct and (len(tweet) <= 280):
        score = (correct + engaging)
    else:
        score = 0
    
    if trace is not None:
        return score >= 2
    return score / 2.0

### 8.4 Evaluation

In [ ]:
# Simple evaluation loop
scores = []
for x in devset:
    pred = program(**x.inputs())
    score = metric(x, pred)
    scores.append(score)

In [ ]:
# Using the built-in Evaluate utility
from dspy.evaluate import Evaluate

# Set up the evaluator
evaluator = Evaluate(
    devset=YOUR_DEVSET,
    num_threads=1,
    display_progress=True,
    display_table=5
)

# Launch evaluation
# evaluator(YOUR_PROGRAM, metric=YOUR_METRIC)

### 8.5 Accessing the Trace

In [ ]:
# Using trace for intermediate step validation
def validate_hops(example, pred, trace=None):
    hops = [example.question] + [outputs.query for *_, outputs in trace if 'query' in outputs]
    
    if max([len(h) for h in hops]) > 100:
        return False
    
    if any(dspy.evaluate.answer_exact_match_str(hops[idx], hops[:idx], frac=0.8) for idx in range(2, len(hops))):
        return False
    
    return True

---

<a id="optimizers"></a>
# 9. Optimizers

Use DSPy optimizers to tune prompts and weights.

### 9.1 Basic Optimizer Usage

In [ ]:
# Using BootstrapFewShotWithRandomSearch optimizer
from dspy.teleprompt import BootstrapFewShotWithRandomSearch

# Set up the optimizer
config = dict(
    max_bootstrapped_demos=4,
    max_labeled_demos=4,
    num_candidate_programs=10,
    num_threads=4
)

teleprompter = BootstrapFewShotWithRandomSearch(metric=YOUR_METRIC_HERE, **config)
optimized_program = teleprompter.compile(YOUR_PROGRAM_HERE, trainset=YOUR_TRAINSET_HERE)

### 9.2 Complete Optimization Example with ReAct

In [ ]:
# Full example: Optimizing a ReAct agent
"""
import dspy
from dspy.datasets import HotPotQA

dspy.configure(lm=dspy.LM('openai/gpt-4o-mini'))

def search(query: str) -> list[str]:
    """Retrieves abstracts from Wikipedia."""
    results = dspy.ColBERTv2(url='http://20.102.90.50:2017/wiki17_abstracts')(query, k=3)
    return [x['text'] for x in results]

# Prepare training data
trainset = [x.with_inputs('question') for x in HotPotQA(train_seed=2024, train_size=500).train]

# Create the program
react = dspy.ReAct("question -> answer", tools=[search])

# Optimize with MIPROv2
tp = dspy.MIPROv2(metric=dspy.evaluate.answer_exact_match, auto="light", num_threads=24)
optimized_react = tp.compile(react, trainset=trainset)
"""

### 9.3 Available Optimizers Overview

In [ ]:
# Optimizers can be accessed via
from dspy.teleprompt import *

# Automatic Few-Shot Learning:
# - LabeledFewShot: Simple few-shot examples from labeled data
# - BootstrapFewShot: Generates demonstrations using a teacher module
# - BootstrapFewShotWithRandomSearch: Random search over demonstrations
# - KNNFewShot: K-Nearest Neighbors for demo selection

# Automatic Instruction Optimization:
# - COPRO: Coordinate ascent for instruction refinement
# - MIPROv2: Bayesian optimization for instructions and demos
# - SIMBA: Stochastic mini-batch sampling for self-reflection
# - GEPA: LM reflection for trajectory improvement

# Automatic Finetuning:
# - BootstrapFinetune: Distills prompt-based programs into weights

# Program Transformations:
# - Ensemble: Combines multiple programs

# Meta-Optimizers:
# - BetterTogether: Combines prompt and weight optimization

### 9.4 Choosing an Optimizer

In [ ]:
# Guidance for choosing optimizers:

# Very few examples (~10): Use BootstrapFewShot
# More data (50+ examples): Try BootstrapFewShotWithRandomSearch
# Instruction optimization only: Use MIPROv2 configured for 0-shot
# Longer optimization runs (40+ trials, 200+ examples): Try MIPROv2
# Efficient program with large LM: Use BootstrapFinetune

### 9.5 Saving and Loading Optimized Programs

In [ ]:
# Save optimized program
# import pickle
# with open('optimized_program.pkl', 'wb') as f:
#     pickle.dump(optimized_program, f)

# Load optimized program
# with open('optimized_program.pkl', 'rb') as f:
#     loaded_program = pickle.load(f)

---

# Summary

This notebook covered all the main topics from the DSPy Learn documentation:

1. **Language Models**: Setup, configuration, direct calls, multiple LMs, usage tracking
2. **Signatures**: Inline and class-based signatures, types, custom types
3. **Modules**: Predict, ChainOfThought, custom modules, usage tracking
4. **Adapters**: ChatAdapter, JSONAdapter, native function calling
5. **Tools**: ReAct, manual tool handling, async tools
6. **MCP**: HTTP and stdio server integration
7. **Data Handling**: Example objects, input keys
8. **Metrics**: Simple metrics, AI feedback metrics, evaluation
9. **Optimizers**: BootstrapFewShot, MIPROv2, and other optimizers

For more information, visit [dspy.ai](https://dspy.ai) and [dspy.ai/learn](https://dspy.ai/learn/).